# 04.R1 — ResNet-50 Dataset Preparation

**Phase 04 — Document Processing & OCR (ResNet-50 Visual Pipeline)**  
**Objective:** Discover valid supporting document images, assign canonical category labels, verify class balance, create deterministic stratified `train` / `val` / `test` partitions (70% / 15% / 15%) with zero file overlap, and export `data/interim/document_processing/resnet50/resnet50_dataset_manifest.csv` plus `models/phase_04/resnet50/label_mapping.json`.


In [ ]:
import os
import sys
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

def _find_root(p: Path = Path.cwd()) -> Path:
    for parent in [p.resolve(), *p.resolve().parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

RESNET_DATA_DIR = paths.interim_doc_proc / 'resnet50'
RESNET_MODEL_DIR = paths.resnet50_model_dir
RESNET_DATA_DIR.mkdir(parents=True, exist_ok=True)
RESNET_MODEL_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_CSV = RESNET_DATA_DIR / 'resnet50_dataset_manifest.csv'
LABEL_MAP_JSON = RESNET_MODEL_DIR / 'label_mapping.json'

SUPPORTED_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IMAGE_ROOT = paths.raw_documents_dir / 'capture'
if not IMAGE_ROOT.exists():
    IMAGE_ROOT = paths.raw_documents_dir

rows = []
for cat_dir in sorted([d for d in IMAGE_ROOT.iterdir() if d.is_dir()]):
    cat_name = cat_dir.name.lower()
    for fpath in sorted(cat_dir.rglob('*')):
        if fpath.is_file() and fpath.suffix.lower() in SUPPORTED_EXTS:
            try:
                with Image.open(fpath) as im:
                    w, h = im.size
                rel = str(fpath.relative_to(paths.root)).replace('\\', '/')
                rows.append({
                    'document_id': f"DOC_{hashlib.md5(rel.encode('utf-8')).hexdigest()[:8].upper()}",
                    'filename': fpath.name,
                    'relative_path': rel,
                    'label_name': cat_name,
                    'width': int(w),
                    'height': int(h),
                    'source_tier': 'PRIMARY',
                })
            except Exception:
                continue

df_ds = pd.DataFrame(rows)
if df_ds.empty:
    raise RuntimeError(f"No readable images found under {IMAGE_ROOT}")

classes = sorted(df_ds['label_name'].unique().tolist())
label2idx = {c: i for i, c in enumerate(classes)}
idx2label = {str(i): c for i, c in enumerate(classes)}
df_ds['label_id'] = df_ds['label_name'].map(label2idx)

# Stratified 70 / 15 / 15 split
train_idx, temp_idx = train_test_split(
    df_ds.index, test_size=0.30, random_state=RANDOM_SEED, stratify=df_ds['label_id']
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=RANDOM_SEED, stratify=df_ds.loc[temp_idx, 'label_id']
)
df_ds['split'] = 'train'
df_ds.loc[val_idx, 'split'] = 'val'
df_ds.loc[test_idx, 'split'] = 'test'

assert set(train_idx).isdisjoint(set(val_idx))
assert set(train_idx).isdisjoint(set(test_idx))
assert set(val_idx).isdisjoint(set(test_idx))

df_ds.to_csv(MANIFEST_CSV, index=False, encoding='utf-8')
with open(LABEL_MAP_JSON, 'w', encoding='utf-8') as f:
    json.dump({'label2idx': label2idx, 'idx2label': idx2label, 'num_classes': len(classes)}, f, indent=2)

print(f"[OK] Saved ResNet-50 dataset manifest ({len(df_ds):,} images, {len(classes)} classes) -> {MANIFEST_CSV}")
display(pd.crosstab(df_ds['label_name'], df_ds['split'], margins=True))
